wget https://github.com/milvus-io/milvus/releases/download/v2.4.6/milvus-standalone-docker-compose.yml -O docker-compose.yml

docker-compose up -d

docker ps

In [1]:
import os
import sys
from pathlib import Path

# 현재 노트북이 있는 경로 기준으로 프로젝트 루트(recsys) 경로 계산
current_dir = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in globals()
    else os.getcwd()
)
project_root = os.path.abspath(os.path.join(current_dir, ".."))

# recsys 폴더를 sys.path에 추가
if project_root not in sys.path:
    sys.path.append(project_root)

from setup_env import setup_env

root_dir = setup_env()

from utils.dataset.config import DatasetPath

[INFO] Font applied: NanumGothic


In [2]:
from pymilvus import (
    connections,
    FieldSchema,
    CollectionSchema,
    DataType,
    Collection,
    utility,
)
import pandas as pd
import numpy as np
from tqdm import tqdm

# -----------------------------------------------
# Milvus 연결
# -----------------------------------------------
print(">>> Connecting to Milvus...")
connections.connect(alias="default", host="localhost", port="19530")
print(">>> Connected to Milvus")

# -----------------------------------------------
# 기존 컬렉션 삭제
# -----------------------------------------------
COLLECTION_NAME = "fashion_items"
if utility.has_collection(COLLECTION_NAME):
    print(
        f">>> Collection '{COLLECTION_NAME}' already exists. Dropping and recreating..."
    )
    utility.drop_collection(COLLECTION_NAME)

# -----------------------------------------------
# 컬렉션 스키마 정의
# -----------------------------------------------
DIM = 768

fields = [
    FieldSchema(name="item_id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="name", dtype=DataType.VARCHAR, max_length=200),
    FieldSchema(name="brand_name", dtype=DataType.VARCHAR, max_length=100),
    FieldSchema(name="price", dtype=DataType.INT64),
    FieldSchema(name="discount_price", dtype=DataType.INT64),
    FieldSchema(name="gender", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="age_group", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="base_color", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="season", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="year", dtype=DataType.INT64),
    FieldSchema(name="usage", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="master_category", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="sub_category", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="article_type", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="fit", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="occasion", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="image_vector", dtype=DataType.FLOAT_VECTOR, dim=DIM),
    FieldSchema(name="text_vector", dtype=DataType.FLOAT_VECTOR, dim=DIM),
]

schema = CollectionSchema(
    fields=fields, description="Fashion item metadata with embeddings"
)
collection = Collection(name=COLLECTION_NAME, schema=schema)
print(f">>> Created collection: '{COLLECTION_NAME}'")

# -----------------------------------------------
# 4️⃣ 인덱스 생성
# -----------------------------------------------
index_params = {
    "metric_type": "IP",
    "index_type": "HNSW",
    "params": {"M": 48, "efConstruction": 200},
}
collection.create_index(field_name="image_vector", index_params=index_params)
print(">>> Index created for 'image_vector'")

index_params = {
    "metric_type": "IP",
    "index_type": "HNSW",
    "params": {"M": 48, "efConstruction": 200},
}
collection.create_index(field_name="text_vector", index_params=index_params)
print(">>> Index created for 'text_vector'")

# -----------------------------------------------
# 데이터 준비
# -----------------------------------------------
dataset_dir = Path(root_dir).joinpath("data/dataset")
paths = DatasetPath(base_dir=dataset_dir, dataset_name="fashion")

df_item_metadata = pd.read_parquet(paths.item_metadata_path)
df_image_vectors = pd.read_parquet(paths.image_vectors_path)
df_text_vectors = pd.read_parquet(paths.text_vectors_path)

df_item_metadata = df_item_metadata.merge(df_image_vectors, how="left", on="item_id")
df_item_metadata = df_item_metadata.merge(df_text_vectors, how="left", on="item_id")
df_item_metadata = df_item_metadata.drop(columns="description")

df_item_metadata["price"] = df_item_metadata["price"].astype(int)
df_item_metadata["discount_price"] = df_item_metadata["discount_price"].astype(int)
df_item_metadata["year"] = df_item_metadata["year"].astype(int)

# -----------------------------------------------
# 데이터 삽입
# -----------------------------------------------
print(">>> Inserting data...")
collection.insert(df_item_metadata.values.T.tolist())
collection.flush()
print(f">>> Inserted {df_item_metadata.shape[0]:,} rows")

# -----------------------------------------------
# 메모리 로드 + 검색 테스트
# -----------------------------------------------
collection.load()
query_vector = np.random.rand(DIM).astype(np.float32)

search_params = {"metric_type": "IP", "params": {"ef": 100}}
results = collection.search(
    data=[query_vector],
    anns_field="image_vector",
    param=search_params,
    limit=5,
    expr='gender == "Women" and master_category == "Apparel"',
    output_fields=[
        "item_id",
        "brand_name",
        "gender",
        "master_category",
        "article_type",
    ],
)

print(">>> Search Results:")
for hit in results[0]:
    print(
        f">>> item_id={hit.entity.get('item_id')}, brand={hit.entity.get('brand_name')}, score={hit.score:.4f}"
    )

print(">>> Milvus setup complete!")

>>> Connecting to Milvus...
>>> Connected to Milvus
>>> Collection 'fashion_items' already exists. Dropping and recreating...
>>> Created collection: 'fashion_items'
>>> Index created for 'image_vector'
>>> Index created for 'text_vector'
>>> Inserting data...
>>> Inserted 10,000 rows
>>> Search Results:
>>> item_id=1498, brand=Flying Machine, score=0.5709
>>> item_id=1795, brand=Myntra, score=0.3891
>>> item_id=7307, brand=Fabindia, score=0.3389
>>> item_id=1051, brand=Angry Birds, score=0.3196
>>> item_id=1229, brand=Fabindia, score=0.2876
>>> Milvus setup complete!
